# Local model on magicstudio — YOLO sizer + Ollama VLM
Everything runs on the Mac Studio (M1 Max, MPS). Heavy files live on `/Volumes/magicLABbox/baebugs`
(venv, weights, datasets, runs); the Ollama vision model lives in `~/.ollama`.

1. Check the environment  2. Build/refresh training data (synthetic now, real later)  3. Train or reuse weights
4. Benchmark mm accuracy: YOLO vs classical vs consensus  5. Analyse one image end-to-end
6. Local VLM QA + label reading (offline Claude stand-in)

**Swap data in the next cell**: point `IMAGE` at an AmScope capture, `BUGPICKER_SCAN` at a BugPicker scan dir
to fine-tune on real trays, `LABEL_PHOTO` at a specimen label.

In [ ]:
# ---- swap your data here ----
ROOT = "/Volumes/magicLABbox/baebugs"
DATA_YAML = f"{ROOT}/datasets/synth_seg/data.yaml"   # or a real dataset from export-bugpicker
TRUTH_JSON = f"{ROOT}/datasets/synth_seg/truth_val.json"
WEIGHTS = f"{ROOT}/runs/baebugs-seg/weights/best.pt"
RETRAIN = False          # True = train again (40 epochs ~ minutes on M1 Max)
EPOCHS = 40
BUGPICKER_SCAN = None    # e.g. "/path/to/BugPicker/scans/scan_20261025_101500" -> fine-tune on real data
IMAGE = None             # AmScope capture; None = a synthetic validation image
PX_PER_MM = None         # set for real images (or use an ArUco marker)
LABEL_PHOTO = None       # specimen label photo for the VLM
VLM = "qwen2.5vl:3b

In [ ]:
import os, sys, json, glob
os.environ.setdefault("YOLO_CONFIG_DIR", f"{ROOT}/.ultralytics")
sys.path.insert(0, "..")
import torch, ultralytics, cv2, numpy as np, matplotlib.pyplot as plt
from baebugs import local_vlm
from baebugs.train import device, fit, bench
print("torch", torch.__version__, "| device:", device(), "| ultralytics", ultralytics.__version__)
print("VLM available:", local_vlm.available(VLM))
print("weights present:", os.path.exists(WEIGHTS))

## 2–3 · Data and training

In [ ]:
if BUGPICKER_SCAN:
    from baebugs.bugpicker_bridge import export_yolo
    DATA_YAML = str(export_yolo(BUGPICKER_SCAN, f"{ROOT}/datasets/bugpicker_real") / "data.yaml")
    RETRAIN = True
if not os.path.exists(DATA_YAML):
    from baebugs.synth_seg import build_dataset
    build_dataset(f"{ROOT}/datasets/synth_seg", 300, 60)
if RETRAIN or not os.path.exists(WEIGHTS):
    # fine-tune from current weights if they exist (keeps what was learned), else from COCO yolo11n-seg
    WEIGHTS = str(fit(DATA_YAML, WEIGHTS if os.path.exists(WEIGHTS) else "yolo11n-seg.pt", EPOCHS))
print(WEIGHTS)
res_png = os.path.join(os.path.dirname(os.path.dirname(WEIGHTS)), "results.png")
if os.path.exists(res_png):
    plt.figure(figsize=(12,6)); plt.imshow(plt.imread(res_png)); plt.axis("off");

## 4 · Sizing benchmark against ground truth (mm)

In [ ]:
b = bench(WEIGHTS, TRUTH_JSON)
print(json.dumps(b, indent=1))

## 5 · One image, end to end: YOLO + classical → consensus → sieve plan

In [ ]:
from baebugs.measure import YoloMeasurer, classical_measure, consensus, annotate
from baebugs.sieve import plan_stack
if IMAGE:
    img, ppm = cv2.imread(IMAGE), PX_PER_MM
else:
    t = json.load(open(TRUTH_JSON))[0]
    img, ppm = cv2.imread(os.path.join(os.path.dirname(TRUTH_JSON), t["image"])), t["px_per_mm"]
A = YoloMeasurer(WEIGHTS)(img, ppm); B = classical_measure(img, ppm)
ms = consensus(A, B, ppm)
print(f"YOLO {len(A)} | classical {len(B)} | consensus {sum(m.flag is None for m in ms)} agreed, {sum(m.flag is not None for m in ms)} flagged")
vis = annotate(img, ms)
plt.figure(figsize=(8,8)); plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); plt.axis("off")
plan = plan_stack([m.width_mm for m in ms if m.flag is None])
print(json.dumps(plan.to_dict(), indent=1))

## 6 · Local VLM: QA review and label reading (offline)

In [ ]:
if local_vlm.available(VLM):
    import time; t0 = time.time()
    qa = local_vlm.review(vis, ms, model=VLM)
    print(f"QA ({time.time()-t0:.1f}s):", json.dumps(qa, indent=1)[:1500])
    if LABEL_PHOTO:
        print("label:", json.dumps(local_vlm.read_label(cv2.imread(LABEL_PHOTO), model=VLM), indent=1))
else:
    print(f"pull the model first: ollama pull {VLM}")

### Next: real data
* Capture 30–50 AmScope frames of the kit analogs (with an ArUco marker for scale), or run BugPicker scans.
* Set `BUGPICKER_SCAN` (or build a dataset with `export_yolo`), `RETRAIN=True`, rerun; the benchmark cell then reports real-data accuracy.
* Every run's weights live under `runs/` on magicLABbox; log the weights hash in provenance before using them on the bench.